# ML-04 — Search Intelligence Data Contract

**Track:** Machine Learning · Foundations  
**Lane:** Refresh / Content Opportunity Scoring  
**Data:** FlyRank internship warehouse (`fact_content_daily_performance`)

This notebook follows the assignment in order: contract → three verification queries → five-feature frame → leakage trap → limitation/self-check.

> **Security:** the Hugging Face READ token is loaded from an environment variable or Colab Secret named `HF_TOKEN`. It is never written into a notebook cell or committed to GitHub.

## 1. Data contract — five plain-language answers

**1. What one row means (grain)**  
One final feature-frame row represents **one pseudonymized content item for one client** (`client_hash_id × content_hash_id`), after aggregating daily warehouse records over the March 2026 feature window.

**2. Which table(s) we use**  
- `fact_content_daily_performance` — daily search-performance measurements.
- `dim_content` — publication/status metadata used only for the analysis universe filter.

**3. Time window**  
- **Feature window:** 2026-03-01 → 2026-03-31.  
- **Decision cutoff:** 2026-03-31.  
- **Label window:** 2026-04-01 → 2026-04-28.  
The windows do not overlap. A feature is legal only if it could be known by the March 31 decision moment.

**4. What we predict**  
`went_dark_apr = 1` when a page with measurable March history records **zero GSC clicks in the April label window**; otherwise 0. This is a measurable outcome proxy for pages that may need review.

**5. One deliberate exclusion**  
Any **April outcome field** is excluded from the March feature set. In particular, April `gsc_clicks` is label information, not a feature, because it is only known after the March 31 decision moment.

## Setup — connect to the gated warehouse

The warehouse is hosted as partitioned Parquet on Hugging Face. DuckDB reads only the March/April partitions needed here; the full warehouse is never loaded into pandas.

In [1]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

print("Connected.")
print("Feature window: March 2026")
print("Label window: April 1–28, 2026")


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Connected.
Feature window: March 2026
Label window: April 1–28, 2026


## 2. Prove three facts with exactly three verification queries

### Query 1 — grain

The warehouse fact table should have at most one row per `report_date × client_hash_id × content_hash_id`. We verify that duplicate groups are zero.

In [2]:
q1 = f"""
SELECT COUNT(*) AS duplicate_groups
FROM (
    SELECT report_date, client_hash_id, content_hash_id
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
) d
"""
q1_result = con.sql(q1).fetchdf()
display(q1_result)
assert q1_result.loc[0, "duplicate_groups"] == 0
print("Grain verified: no duplicate date × client × content groups.")

,duplicate_groups
0,0


Grain verified: no duplicate date × client × content groups.


### Query 2 — March row count and date span

This records the actual size and date coverage of the mid-panel March slice.

In [3]:
q2 = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM {FACT_MAR}
"""
q2_result = con.sql(q2).fetchdf()
display(q2_result)
assert str(q2_result.loc[0, "first_date"])[:10] == "2026-03-01"
assert str(q2_result.loc[0, "last_date"])[:10] == "2026-03-31"
print("March date span verified.")

,row_count,clients,content_items,first_date,last_date,distinct_dates
0,9841378,55,331437,2026-03-01,2026-03-31,31


March date span verified.


### Query 3 — availability with `IS TRUE`

`gsc_data_available` is a measurement-availability flag. We explicitly use `IS TRUE` rather than treating NULL or FALSE as measured zero traffic. The output shows how many March rows survive the availability filter.

In [4]:
q3 = f"""
SELECT COUNT(*) AS available_rows
FROM {FACT_MAR}
WHERE gsc_data_available IS TRUE
"""
q3_result = con.sql(q3).fetchdf()
display(q3_result)
assert q3_result.loc[0, "available_rows"] > 0
print("Availability check passed: only rows with gsc_data_available IS TRUE are used for measured GSC features.")

,available_rows
0,3611061


Availability check passed: only rows with gsc_data_available IS TRUE are used for measured GSC features.


## 3. Five-feature frame from the same March month

All five features are computed only from March measurements, so they are available at the March 31 decision moment.

In [5]:
feature_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    march_impressions,
    march_clicks,
    march_clicks / NULLIF(march_impressions, 0) AS ctr_march,
    avg_position,
    days_with_impressions
FROM march
WHERE march_impressions >= 100
  AND march_clicks >= 3
""").df()
display(feature_frame.head(10))
print(f"Feature-frame rows: {len(feature_frame):,}")

,client_hash_id,content_hash_id,march_impressions,march_clicks,ctr_march,avg_position,days_with_impressions
0,client_ff644d8251367cbb,content_49ee3c7d8c1805cc,1278.0,4.0,0.003130,9.266041,31
1,client_ff644d8251367cbb,content_554d4a899e83b46e,2701.0,13.0,0.004813,6.629026,31
2,client_ff644d8251367cbb,content_4ada171d9bfee230,1972.0,9.0,0.004564,8.463489,31
3,client_b10cb2997d0c7c86,content_9c308382f7af9b21,6970.0,24.0,0.003443,3.392683,31
4,client_b10cb2997d0c7c86,content_f0bb90dca9722fba,288.0,3.0,0.010417,7.750000,28
5,client_a80fca3f171ed1de,content_44d344390934eef3,9310.0,49.0,0.005263,4.293555,31
6,client_a80fca3f171ed1de,content_42b6afd8fa47c5ba,186.0,4.0,0.021505,5.634409,26
7,client_a80fca3f171ed1de,content_cbf14fa01f6e3080,2161.0,4.0,0.001851,5.019898,31
8,client_a80fca3f171ed1de,content_32b30a47c4e8cc2b,230.0,4.0,0.017391,3.360870,31
9,client_a80fca3f171ed1de,content_47b9e3d74d302801,307.0,5.0,0.016287,5.732899,31


Feature-frame rows: 38,459


### Feature availability — “knowable at the decision moment because…”

1. **`march_impressions`** — knowable because it is the sum of measured GSC impressions through March 31.
2. **`march_clicks`** — knowable because it is the sum of measured GSC clicks through March 31.
3. **`ctr_march`** — knowable because it is calculated from March clicks and impressions only.
4. **`avg_position`** — knowable because it uses impression-weighted March search position only.
5. **`days_with_impressions`** — knowable because it counts March days with measured search impressions.

**Excluded from the feature frame:** April clicks/impressions and the derived April label, because those occur after the decision cutoff.

## 4. Leakage trap — deliberately add one label-derived column

The honest model uses only the five March features. Then we deliberately add `went_dark_apr`, which is derived from the April outcome. That column would not exist at the March 31 decision moment.

The experiment is educational: the leaked score should become unrealistically strong. We then remove the leaked column and keep the honest score.

In [6]:
label_frame = con.sql(f"""
WITH april AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM {FACT_APR}
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-28'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT
    f.*,
    CASE WHEN COALESCE(a.april_clicks, 0) = 0 THEN 1 ELSE 0 END AS went_dark_apr
FROM (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_march,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100 AND SUM(gsc_clicks) >= 3
) f
LEFT JOIN april a USING (client_hash_id, content_hash_id)
""").df()

feature_cols = [
    "march_impressions",
    "march_clicks",
    "ctr_march",
    "avg_position",
    "days_with_impressions",
]

model_df = label_frame.dropna(subset=feature_cols + ["went_dark_apr"]).copy()
X = model_df[feature_cols]
y = model_df["went_dark_apr"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

honest_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
honest_model.fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])

# THE TRAP: add the label itself as a feature.
X_leaked = model_df[feature_cols + ["went_dark_apr"]]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaked, y, test_size=0.30, random_state=42, stratify=y
)
leaked_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
leaked_model.fit(X_train_l, y_train_l)
leaked_auc = roc_auc_score(y_test_l, leaked_model.predict_proba(X_test_l)[:, 1])

print(f"Honest ROC-AUC (5 March features): {honest_auc:.4f}")
print(f"Leaked ROC-AUC (+ April label):     {leaked_auc:.4f}")

# Remove the trap: production keeps only the five honest features.
final_feature_cols = feature_cols
print("Final feature set after removing leakage:", final_feature_cols)

Honest ROC-AUC (5 March features): 0.7748
Leaked ROC-AUC (+ April label):     1.0000
Final feature set after removing leakage: ['march_impressions', 'march_clicks', 'ctr_march', 'avg_position', 'days_with_impressions']


## 5. One named limitation

**Limitation:** the March → April experiment covers one calendar transition only. Search demand can be seasonal, and client history is an unbalanced panel, so a single month-to-month result should be treated as a decision-support experiment rather than evidence that the relationship will generalize across all clients or seasons.

## Submission self-check

- [ ] Run **Runtime → Run all** in Colab after adding `HF_TOKEN` to Colab Secrets.
- [ ] Confirm all code cells have visible outputs.
- [ ] Confirm the three verification queries show their outputs.
- [ ] Confirm the leakage experiment shows both honest and leaked scores.
- [ ] Confirm the token is **not** present anywhere in the notebook.
- [ ] Save the executed notebook to `work/notebooks/w03_data_contract.ipynb`.
- [ ] Commit and push it to your GitHub repo.
- [ ] Submit the repository URL on the FlyRank assignment card.